In [ ]:
import sys; sys.path.insert(0, "..")  # or absolute path to project root
from src.pipelines.silver.pipeline import SilverPipeline
from src.pipelines.gold.pipeline import GoldPipeline
from src.storage.local_storage_backend import LocalStorageBackend
from src.models.position.train import train_model
from src.config.paths import LOCAL_DATA_DIR

#storage = LocalStorageBackend(LOCAL_DATA_DIR)

#silver_pipeline = SilverPipeline(storage)
#silver_pipeline.build_silver_data()

#gold_pipeline = GoldPipeline(storage)
#gold_pipeline.build_gold_data()

train_model()


Baseline MAE (predict position = grid): 2.71
Model MAE: 2.11


In [ ]:
import sys; sys.path.insert(0, "..")  # or absolute path to project root

import joblib
import pandas as pd
from src.models.position.train import load_training_data
from src.config.paths import LOCAL_MODELS_DIR

train, test, features = load_training_data()
test = test.dropna(subset=features)
test = test[test["status"].isin(["Finished", "Lapped", "+1 Lap", "+2 Laps"])]

model = joblib.load(LOCAL_MODELS_DIR / "model_v2.pkl")

x_test = test[features]
test = test.copy()
test["predicted"] = model.predict(x_test)
test["actual"] = test["position"]
test["error"] = test["predicted"] - test["actual"]
test["abs_error"] = test["error"].abs()
test["grid_abs_error"] = (test["grid"] - test["actual"]).abs()
test["beats_grid"] = test["abs_error"] < test["grid_abs_error"]

display_cols = [
    "season", "round", "driverId", "circuitName",
    "grid", "predicted", "actual", "error", "abs_error", "beats_grid", "status"
]
print(f"Model MAE: {test['abs_error'].mean():.2f}")
print(f"Grid MAE:  {test['grid_abs_error'].mean():.2f}")
print(f"Beats grid: {test['beats_grid'].mean():.1%}")

(
    test[display_cols]
    .sort_values("abs_error", ascending=False)
    .head(40)
    .style.format({
        "predicted": "{:.1f}",
        "actual": "{:.0f}",
        "error": "{:+.1f}",
        "abs_error": "{:.1f}",
    })
)

Model MAE: 2.11
Grid MAE:  2.71
Beats grid: 54.6%


,season,round,driverId,circuitName,grid,predicted,actual,error,abs_error,beats_grid,status
986,2024,22,hamilton,Las Vegas Strip Street Circuit,10.000000,9.5,2,+7.5,7.5,True,Finished
855,2024,24,piastri,Yas Marina Circuit,2.000000,2.5,10,-7.5,7.5,True,Finished
304,2024,21,gasly,Autódromo José Carlos Pace,13.000000,10.5,3,+7.5,7.5,True,Finished
990,2024,24,hamilton,Yas Marina Circuit,16.000000,11.3,4,+7.3,7.3,True,Finished
1021,2024,15,tsunoda,Circuit Park Zandvoort,11.000000,10.3,17,-6.7,6.7,False,Lapped
852,2024,23,norris,Losail International Circuit,3.000000,3.4,10,-6.6,6.6,True,Finished
440,2024,21,alonso,Autódromo José Carlos Pace,9.000000,7.9,14,-6.1,6.1,False,Finished
301,2024,19,ocon,Circuit of the Americas,12.000000,11.9,18,-6.1,6.1,False,Lapped
307,2024,22,ocon,Las Vegas Strip Street Circuit,11.000000,11.1,17,-5.9,5.9,True,Lapped
582,2024,24,leclerc,Yas Marina Circuit,19.000000,8.9,3,+5.9,5.9,True,Finished
